# Challenge 1: Specialized Agents with Tools

## What We're Building

Instead of one agent doing everything badly, we'll create **5 specialized agents** that each:
- Have a focused responsibility
- Own specific infrastructure tools
- Produce structured outputs for the next agent

| Agent | Role | Tools |
|-------|------|-------|
| **Triage** | Classify, check history, get runbook | `check_alert_history`, `get_runbook` |
| **Diagnostics** | Find root cause using metrics/logs | `get_metrics`, `get_logs`, `check_dependencies` |
| **Remediation** | Execute the fix | `restart_pod`, `scale_service`, `flush_cache`, `toggle_feature_flag`, `escalate_to_human` |
| **Verification** | Confirm fix worked | `get_health_status`, `run_smoke_test` |
| **Communications** | Notify team, create tickets | `post_to_slack`, `create_incident_ticket`, `update_status_page` |

---

## How This Challenge Works

1. The **Triage Agent** is provided as a complete reference
2. You build the remaining **4 agents** yourself
3. Each agent has a validation cell to check your work

> **Tip**: Read the Triage Agent carefully — it shows the pattern for all agents.

In [ ]:
import os
import sys
import json
sys.path.insert(0, "..")

from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket, update_status_page,
)

load_dotenv("../.env")

with open("../data/incidents.json") as f:
    incidents = json.load(f)

alert = incidents[0]  # payment-api high latency
print(f"\U0001f534 Working with: {alert['title']}")
print(f"   Service: {alert['service']} | Severity: {alert['severity']}")
print(f"   Description: {alert['description']}")

---
## Agent 1: Triage Agent (REFERENCE — provided for you)

Study this agent. It demonstrates the pattern:
1. Clear role in `instructions`
2. Tool usage guidance (which tool, when)
3. Output format specification
4. Constraints (what NOT to do)

### Expected Output
```
TRIAGE SUMMARY
==============
Severity: Critical
Recurring: Yes — 3 similar alerts in last 7 days
Runbook: high_latency playbook found
Auto-remediation: Allowed
Recommended action: Proceed to diagnostics → likely memory leak based on history
```

In [ ]:
async def build_triage_agent():
    """REFERENCE IMPLEMENTATION — study this pattern, then build the next 4 agents."""
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        triage_agent = client.create_agent(
            name="TriageAgent",
            instructions="""You are an incident Triage Agent — the first responder when an alert fires.

When given an alert, you MUST:
1. Call check_alert_history with the service name to see if this is recurring
2. Call get_runbook with the incident type (e.g., "high_latency") to find the playbook

Then produce a TRIAGE SUMMARY with:
- Severity assessment (critical/high/medium/low)
- Whether this is a recurring issue (from alert history)
- Whether automated remediation is allowed (from the runbook)
- Recommended next steps for the Diagnostics Agent

CONSTRAINTS:
- Do NOT guess at root causes — that's the Diagnostics Agent's job
- Do NOT suggest fixes — that's the Remediation Agent's job
- ONLY use data from your tools, never make up information""",
            tools=[check_alert_history, get_runbook],
        )

        result = await triage_agent.run(
            f"New alert fired:\n"
            f"Title: {alert['title']}\n"
            f"Service: {alert['service']}\n"
            f"Severity: {alert['severity']}\n"
            f"Description: {alert['description']}\n\n"
            f"Triage this incident now."
        )
        print("\n\U0001f4cb TRIAGE RESULT:\n")
        print(result.text)
        return result.text

triage_output = await build_triage_agent()

---
## Agent 2: Diagnostics Agent (YOUR TURN)

The Diagnostics Agent investigates the root cause using monitoring data.

### Tools Available
| Tool | What it returns |
|------|----------------|
| `get_metrics(service_name)` | CPU, memory, latency, error rate per pod |
| `get_logs(service_name)` | Recent error log entries with timestamps |
| `check_dependencies(service_name)` | Health status of upstream/downstream services |

### Requirements
Your agent must:
- Call ALL THREE tools to gather evidence
- Correlate findings (e.g., high memory + OOM in logs = memory leak)
- Output a **specific** root cause with evidence (not generic advice)
- Include the exact pod/component name that's failing

### Expected Output
```
ROOT CAUSE ANALYSIS
===================
Root cause: Memory leak on payment-api-pod-3
Evidence:
  - Metrics: pod-3 memory at 3891MB/4096MB limit, 4 restarts in last hour
  - Logs: OOMKilled errors at 02:38, 02:41
  - Dependencies: order-service seeing connection timeouts to payment-api
Blast radius: order-service affected (cascading errors)
Recommended fix: Restart payment-api-pod-3 to clear memory leak
```

### Your Task
Fill in the `instructions` and `tools` below:

In [ ]:
async def build_diagnostics_agent(triage_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        diagnostics_agent = client.create_agent(
            name="DiagnosticsAgent",
            # ╔══════════════════════════════════════════════════════╗
            # ║  YOUR CODE: Write the instructions                  ║
            # ║  - State the agent's role                           ║
            # ║  - Tell it which tools to call and in what order    ║
            # ║  - Specify the output format (see Expected Output)  ║
            # ║  - Add constraints (evidence-based, no guessing)    ║
            # ╚══════════════════════════════════════════════════════╝
            instructions="""
TODO: Write your instructions here
""",
            # ╔══════════════════════════════════════════════════════╗
            # ║  YOUR CODE: Provide the correct tools list          ║
            # ║  Which 3 tools does the diagnostics agent need?     ║
            # ╚══════════════════════════════════════════════════════╝
            tools=[],  # TODO: Add the right tools
        )

        result = await diagnostics_agent.run(
            f"Investigate this incident. Triage summary:\n{triage_context}\n\n"
            f"Service: {alert['service']}\n"
            f"Description: {alert['description']}\n\n"
            f"Find the root cause using your monitoring tools."
        )
        print("\n\U0001f50d DIAGNOSTICS RESULT:\n")
        print(result.text)
        return result.text

diagnostics_output = await build_diagnostics_agent(triage_output)

In [ ]:
# ✅ VALIDATION: Check your diagnostics agent worked
assert diagnostics_output, "Diagnostics agent returned empty output!"
output_lower = diagnostics_output.lower()

checks = {
    "Identified pod-3": "pod-3" in output_lower or "pod3" in output_lower,
    "Found memory issue": "memory" in output_lower or "oom" in output_lower,
    "Mentioned restart": "restart" in output_lower,
    "Has evidence": any(w in output_lower for w in ["3891", "4096", "oomkilled", "oom"]),
}

for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

if all(checks.values()):
    print("\n\U0001f389 Diagnostics agent is working correctly!")
else:
    print("\n\u26a0\ufe0f Some checks failed. Review your instructions — is the agent calling all 3 tools?")

---
## Agent 3: Remediation Agent (YOUR TURN)

The Remediation Agent takes **action** on infrastructure. This is the "dangerous" agent.

### Tools Available
| Tool | What it does | When to use |
|------|-------------|-------------|
| `restart_pod(service, pod_id)` | Restarts a specific pod | OOM/memory leak |
| `scale_service(service, replicas)` | Changes replica count | High CPU/traffic |
| `flush_cache(service)` | Clears the cache | Stale data issues |
| `toggle_feature_flag(flag_name, enabled)` | Enables/disables a feature | Failover/degradation |
| `escalate_to_human(reason)` | Pages a human engineer | Can't auto-fix safely |

### Requirements
Your agent must:
- Act ONLY on findings from the diagnostics (not guess)
- Use **exact names** from diagnostics (e.g., `pod-3`, not "the problematic pod")
- Escalate if auto-remediation isn't safe
- Report what action was taken and the result

### Expected Output
```
REMEDIATION ACTIONS
===================
Action taken: Restarted payment-api-pod-3
Result: Pod restarted successfully, new pod healthy
Confidence: High — diagnostics clearly showed OOM on pod-3
```

### Your Task
Build the complete agent (instructions + tools + prompt):

In [ ]:
async def build_remediation_agent(diagnostics_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # ╔══════════════════════════════════════════════════════════════╗
        # ║  YOUR CODE: Build the entire remediation agent              ║
        # ║  - Write instructions (role, tool selection logic, safety)  ║
        # ║  - Choose the right tools                                   ║
        # ║  - Craft the prompt (pass in diagnostics context)           ║
        # ╚══════════════════════════════════════════════════════════════╝
        
        remediation_agent = client.create_agent(
            name="RemediationAgent",
            instructions="""TODO: Write your instructions here""",
            tools=[],  # TODO: Add the right tools
        )

        result = await remediation_agent.run(
            # TODO: Write your prompt here
            # Include the diagnostics findings and service name
            f"TODO: Write your prompt"
        )
        print("\n\U0001f527 REMEDIATION RESULT:\n")
        print(result.text)
        return result.text

remediation_output = await build_remediation_agent(diagnostics_output)

In [ ]:
# ✅ VALIDATION: Check your remediation agent
assert remediation_output, "Remediation agent returned empty output!"
output_lower = remediation_output.lower()

checks = {
    "Took action (restart)": "restart" in output_lower,
    "Targeted pod-3": "pod-3" in output_lower or "pod3" in output_lower,
    "Mentioned result": any(w in output_lower for w in ["success", "completed", "restarted", "healthy"]),
    "Didn't escalate (auto-fix was allowed)": "escalat" not in output_lower,
}

for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

if all(checks.values()):
    print("\n\U0001f389 Remediation agent is working correctly!")
else:
    print("\n\u26a0\ufe0f Some checks failed. Is the agent using exact pod names from diagnostics?")

---
## Agent 4: Verification Agent (YOUR TURN)

After remediation, we need **proof** the fix worked — not just a hopeful "should be fine."

### Tools Available
| Tool | What it returns |
|------|----------------|
| `get_health_status(service)` | Overall health: latency, error rate, pod status |
| `run_smoke_test(service)` | Functional test results: pass/fail for key endpoints |

### Requirements
Your agent must:
- Call BOTH tools (health check AND smoke test)
- Give a clear **verdict**: `RESOLVED` or `NEEDS_ESCALATION`
- Back up the verdict with data

### Expected Output
```
VERIFICATION REPORT
===================
Health Status: All pods healthy, p95 latency 180ms (within threshold)
Smoke Tests: 5/5 passing (checkout, refund, balance, webhook, auth)

VERDICT: RESOLVED
```

### Your Task

In [ ]:
async def build_verification_agent(remediation_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # ╔══════════════════════════════════════════════════════════════╗
        # ║  YOUR CODE: Build the verification agent                    ║
        # ║  - Instructions: Check health + run tests, give verdict     ║
        # ║  - Tools: Which 2 tools verify the fix?                     ║
        # ║  - Prompt: Tell it what was fixed so it knows what to check ║
        # ╚══════════════════════════════════════════════════════════════╝
        
        verification_agent = client.create_agent(
            name="VerificationAgent",
            instructions="""TODO: Write your instructions here""",
            tools=[],  # TODO: Add the right tools
        )

        result = await verification_agent.run(
            # TODO: Write your prompt
            f"TODO: Write your prompt"
        )
        print("\n\u2705 VERIFICATION RESULT:\n")
        print(result.text)
        return result.text

verification_output = await build_verification_agent(remediation_output)

In [ ]:
# ✅ VALIDATION: Check your verification agent
assert verification_output, "Verification agent returned empty output!"
output_lower = verification_output.lower()

checks = {
    "Has a verdict": "resolved" in output_lower or "escalat" in output_lower,
    "Checked health": any(w in output_lower for w in ["health", "latency", "pod"]),
    "Ran smoke tests": any(w in output_lower for w in ["smoke", "test", "pass"]),
    "Verdict is RESOLVED (fix should work)": "resolved" in output_lower,
}

for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

if all(checks.values()):
    print("\n\U0001f389 Verification agent is working correctly!")
else:
    print("\n\u26a0\ufe0f Some checks failed. Is the agent calling BOTH health check and smoke test?")

---
## Agent 5: Communications Agent (YOUR TURN)

The final agent handles all human notifications. It doesn't fix anything — it reports.

### Tools Available
| Tool | What it does |
|------|-------------|
| `post_to_slack(channel, message)` | Posts to a Slack channel |
| `create_incident_ticket(title, description, severity)` | Creates a JIRA/PagerDuty ticket |
| `update_status_page(service, status)` | Updates the public status page |

### Requirements
Your agent must call ALL THREE tools:
1. Slack: Brief summary (what happened, root cause, fix, status) — max 4 lines
2. Ticket: Full timeline for post-mortem
3. Status page: Updated to 'operational'

### Expected Output
```
COMMUNICATIONS SENT
===================
Slack (#incidents): Posted 4-line summary
Ticket: INC-2024-001 created with full timeline
Status page: payment-api → operational
```

### Your Task

In [ ]:
async def build_comms_agent(triage_ctx, diagnostics_ctx, remediation_ctx, verification_ctx):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # ╔══════════════════════════════════════════════════════════════╗
        # ║  YOUR CODE: Build the communications agent                  ║
        # ║  - Instructions: Notify via all 3 channels                  ║
        # ║  - Tools: All 3 communication tools                         ║
        # ║  - Prompt: Pass the full incident timeline                  ║
        # ╚══════════════════════════════════════════════════════════════╝
        
        comms_agent = client.create_agent(
            name="CommunicationsAgent",
            instructions="""TODO: Write your instructions here""",
            tools=[],  # TODO: Add the right tools
        )

        result = await comms_agent.run(
            # TODO: Write your prompt — include ALL context from the pipeline
            f"TODO: Write your prompt"
        )
        print("\n\U0001f4e2 COMMUNICATIONS RESULT:\n")
        print(result.text)
        return result.text

comms_output = await build_comms_agent(triage_output, diagnostics_output, remediation_output, verification_output)

In [ ]:
# ✅ VALIDATION: Check your communications agent
assert comms_output, "Communications agent returned empty output!"
output_lower = comms_output.lower()

checks = {
    "Posted to Slack": "slack" in output_lower or "#incidents" in output_lower or "posted" in output_lower,
    "Created ticket": "ticket" in output_lower or "inc-" in output_lower or "created" in output_lower,
    "Updated status page": "status" in output_lower and "operational" in output_lower,
}

for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

if all(checks.values()):
    print("\n\U0001f389 Communications agent is working correctly!")
else:
    print("\n\u26a0\ufe0f Some checks failed. Did the agent call all 3 communication tools?")

---
## Full Pipeline Summary

Run this cell to see the complete incident response flow:

In [ ]:
print("\n" + "="*60)
print("\U0001f3c1 FULL INCIDENT RESPONSE — PAYMENT-API")
print("="*60)
print(f"\n\U0001f4cb TRIAGE:\n{triage_output[:300]}")
print(f"\n\U0001f50d DIAGNOSTICS:\n{diagnostics_output[:300]}")
print(f"\n\U0001f527 REMEDIATION:\n{remediation_output[:300]}")
print(f"\n\u2705 VERIFICATION:\n{verification_output[:300]}")
print(f"\n\U0001f4e2 COMMUNICATIONS:\n{comms_output[:300]}")
print("\n" + "="*60)
print("\U0001f389 Incident resolved by 5 specialized agents!")
print("="*60)

---
## \U0001f680 Stretch Goal: Try Incident #3

Run your agents on the notification-service incident. The diagnostics should find
a DIFFERENT root cause (rate limiting, not OOM) and the remediation agent should
take a DIFFERENT action (toggle feature flag, not restart pod).

This proves your agents are reasoning, not just pattern-matching.

In [ ]:
# STRETCH: Run on incident #3 — notification-service
alert = incidents[2]  # Switch to notification-service
print(f"\U0001f534 NEW INCIDENT: {alert['title']}")
print(f"   Service: {alert['service']} | Severity: {alert['severity']}")
print(f"   Description: {alert['description']}")
print("\nRunning full pipeline...\n")

triage_2 = await build_triage_agent()
diag_2 = await build_diagnostics_agent(triage_2)
remed_2 = await build_remediation_agent(diag_2)
verify_2 = await build_verification_agent(remed_2)
comms_2 = await build_comms_agent(triage_2, diag_2, remed_2, verify_2)

# Check that it made different decisions
print("\n" + "="*60)
print("\U0001f9ea DID THE AGENT MAKE DIFFERENT DECISIONS?")
print("="*60)
checks = {
    "NOT restart (different root cause)": "restart" not in remed_2.lower() or "feature" in remed_2.lower(),
    "Toggle feature flag OR different action": any(w in remed_2.lower() for w in ["feature_flag", "toggle", "backup", "failover"]),
}
for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

---
## What You Achieved

| What | Single Agent (Basic Approach) | Your Agents (Challenge 1) |
|------|--------------------------|---------------------------|
| Tools | None | 15 real infrastructure APIs |
| Diagnosis | Guesses | Evidence-based (metrics + logs) |
| Fix | Suggests | Executes (restart, scale, toggle) |
| Verification | None | Health checks + smoke tests |
| Communication | None | Slack + ticket + status page |
| Different incidents | Same generic response | Different actions per root cause |

**But there's a problem:** You're manually passing outputs between agents.
What if verification fails? What if you want this to run without you at 3 AM?

---

## \u27a1\ufe0f Next: Challenge 2 — Workflow Orchestration

Wire these agents into an automated pipeline with conditional routing and retry logic.

# 🛠️ Step 2: Specialized Agents with Tools

## What we're building

Instead of one agent doing everything badly, we'll create **specialized agents** that each:
- Have a focused responsibility
- Own specific tools (infrastructure APIs)
- Produce structured outputs for the next agent

## Our Agents

| Agent | Role | Tools |
|-------|------|-------|
| **Triage Agent** | Classify severity, check history, get runbook | `check_alert_history`, `get_runbook` |
| **Diagnostics Agent** | Find root cause using metrics and logs | `get_metrics`, `get_logs`, `check_dependencies` |
| **Remediation Agent** | Execute the fix | `restart_pod`, `scale_service`, `flush_cache`, `toggle_feature_flag` |
| **Verification Agent** | Confirm the fix worked | `get_health_status`, `run_smoke_test` |
| **Comms Agent** | Notify team, create tickets | `post_to_slack`, `create_incident_ticket`, `update_status_page` |

---

## 💡 Key Concept: Task Decomposition

Each agent is an **expert** at one thing. This means:
- Simpler instructions (less confusion)
- Focused tool sets (less chance of misuse)
- Testable in isolation
- Replaceable/upgradable independently

In [ ]:
import os
import json
from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

# Import our mock infrastructure tools
import sys
sys.path.insert(0, "..")

from tools.mock_infra import (
    # Triage tools
    check_alert_history,
    get_runbook,
    # Diagnostics tools
    get_metrics,
    get_logs,
    check_dependencies,
    # Remediation tools
    restart_pod,
    scale_service,
    flush_cache,
    toggle_feature_flag,
    # Verification tools
    get_health_status,
    run_smoke_test,
    # Communications tools
    post_to_slack,
    create_incident_ticket,
    update_status_page,
load_dotenv("../.env")
)

with open("../data/incidents.json") as f:

# Load incident data
with open("data/incidents.json") as f:
    incidents = json.load(f)


alert = incidents[0]  # payment-api high latencyprint(f"   Service: {alert['service']} | Severity: {alert['severity']}")
print(f"🔴 Working with alert: {alert['title']}")

## Agent 1: Triage Agent

The Triage Agent is the first responder. Its job:
1. Check if this is a known/recurring issue
2. Look up the relevant runbook
3. Classify whether automated remediation is possible

### 🎯 YOUR TASK
Fill in the `instructions` for the Triage Agent. It should:
- Use `check_alert_history` to see if this happened before
- Use `get_runbook` to find the remediation playbook
- Output a clear triage summary with: severity, whether it's recurring, and recommended action

In [ ]:
async def build_triage_agent():
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Triage Agent
        # Hint: Tell it its role, what tools it has, and what output format you expect
        triage_instructions = """
You are an incident Triage Agent. Your job is to be the first responder when an alert fires.

When given an alert:
1. Use check_alert_history to see if this service has had similar issues recently
2. Use get_runbook to look up the standard operating procedure for this type of incident
3. Produce a triage summary with:
   - Severity assessment (critical/high/medium/low)
   - Whether this is a recurring issue
   - Whether automated remediation is allowed (from the runbook)
   - Recommended next steps for the Diagnostics Agent

Be concise and factual. Do not guess at root causes — that's the Diagnostics Agent's job.
"""

        triage_agent = client.create_agent(
            name="TriageAgent",
            instructions=triage_instructions,
            tools=[check_alert_history, get_runbook],
        )
        
        # Run the triage agent on our alert
        triage_prompt = f"""New alert received:
Title: {alert['title']}
Service: {alert['service']}
Severity: {alert['severity']}
Description: {alert['description']}

Triage this incident."""

        result = await triage_agent.run(triage_prompt)
        print("\n📋 TRIAGE RESULT:\n")
        print(result.text)
        return result.text

triage_output = await build_triage_agent()

## Agent 2: Diagnostics Agent

The Diagnostics Agent digs into the technical details. Its job:
1. Pull real metrics (CPU, memory, latency, error rates)
2. Analyze logs for error patterns
3. Check dependency health
4. Identify the root cause

### 🎯 YOUR TASK
Fill in the `instructions` for the Diagnostics Agent. It should:
- Use its tools to gather evidence (not guess!)
- Correlate findings across metrics, logs, and dependencies
- Output a root cause analysis with specific evidence

In [ ]:
async def build_diagnostics_agent(triage_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Diagnostics Agent
        # Hint: It should systematically gather evidence using all 3 tools
        diagnostics_instructions = """
You are an incident Diagnostics Agent. Your job is to identify the root cause of an incident.

You have access to real monitoring data. When investigating:
1. Use get_metrics to check CPU, memory, latency, and error rates for the affected service
2. Use get_logs to find error messages and patterns
3. Use check_dependencies to see if upstream/downstream services are healthy

Your output should be a root cause analysis with:
- What is failing (specific pod, specific component)
- Why it's failing (evidence from metrics/logs)
- What's the blast radius (which other services are affected)
- Recommended remediation action with specific parameters
  (e.g., "restart payment-api-pod-3" not just "restart a pod")

IMPORTANT: Base your analysis ONLY on data from your tools. Do not speculate.
"""

        diagnostics_agent = client.create_agent(
            name="DiagnosticsAgent",
            instructions=diagnostics_instructions,
            tools=[get_metrics, get_logs, check_dependencies],
        )

        # Pass triage context + original alert
        diag_prompt = f"""Investigate this incident. The Triage Agent has already assessed it:

--- TRIAGE SUMMARY ---
{triage_context}

--- ORIGINAL ALERT ---
Service: {alert['service']}
Description: {alert['description']}

Run diagnostics and identify the root cause."""

        result = await diagnostics_agent.run(diag_prompt)
        print("\n🔍 DIAGNOSTICS RESULT:\n")
        print(result.text)
        return result.text

diagnostics_output = await build_diagnostics_agent(triage_output)

## Agent 3: Remediation Agent

The Remediation Agent takes action. It has the "dangerous" tools that actually change infrastructure.

### 🎯 YOUR TASK
Fill in the `instructions` for the Remediation Agent. Key considerations:
- It should ONLY act based on the diagnostics findings (not guess)
- It should check the runbook's `auto_remediation_allowed` flag
- If auto-remediation isn't allowed, it should call `escalate_to_human` instead

In [ ]:
async def build_remediation_agent(diagnostics_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Remediation Agent
        remediation_instructions = """
You are an incident Remediation Agent. Your job is to execute fixes based on the diagnosis.

You have access to infrastructure tools:
- restart_pod: Restart a specific pod (use when OOM/memory leak detected)
- scale_service: Scale to more replicas (use when CPU is high or traffic spike)
- flush_cache: Flush a cache (use when stale data suspected)
- toggle_feature_flag: Enable/disable features (use for graceful degradation)
- escalate_to_human: Page a human when auto-fix isn't possible

Rules:
1. ONLY take actions that the diagnostics directly support
2. If the runbook says auto_remediation_allowed=False, call escalate_to_human
3. Be specific: use exact pod names, exact service names, exact flag names from the diagnostics
4. Report what actions you took and their results

Do NOT take multiple unrelated actions. Fix the root cause identified by diagnostics.
"""

        remediation_agent = client.create_agent(
            name="RemediationAgent",
            instructions=remediation_instructions,
            tools=[restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human],
        )

        remediation_prompt = f"""Execute remediation based on this diagnosis:

--- DIAGNOSTICS FINDINGS ---
{diagnostics_context}

--- SERVICE ---
{alert['service']}

Take the appropriate remediation action now."""

        result = await remediation_agent.run(remediation_prompt)
        print("\n🔧 REMEDIATION RESULT:\n")
        print(result.text)
        return result.text

remediation_output = await build_remediation_agent(diagnostics_output)

## Agent 4: Verification Agent

After remediation, we need to verify the fix actually worked.

### 🎯 YOUR TASK
Create the Verification Agent. It should:
- Check service health after the fix
- Run smoke tests
- Report whether the incident is resolved or needs escalation

In [ ]:
async def build_verification_agent(remediation_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write the instructions and create the agent
        verification_instructions = """
You are a Verification Agent. After remediation actions are taken, you verify the fix worked.

Steps:
1. Use get_health_status to check if the service is now healthy
2. Use run_smoke_test to run functional tests against the service
3. Report your findings:
   - Is the service healthy? (latency, error rate, resource usage)
   - Are smoke tests passing?
   - Final verdict: RESOLVED or NEEDS_ESCALATION

If all checks pass, mark as RESOLVED.
If any checks fail, mark as NEEDS_ESCALATION with details.
"""

        verification_agent = client.create_agent(
            name="VerificationAgent",
            instructions=verification_instructions,
            tools=[get_health_status, run_smoke_test],
        )

        verify_prompt = f"""The following remediation was performed:

--- REMEDIATION ACTIONS ---
{remediation_context}

--- SERVICE ---
{alert['service']}

Verify that the service is now healthy and the incident is resolved."""

        result = await verification_agent.run(verify_prompt)
        print("\n✅ VERIFICATION RESULT:\n")
        print(result.text)
        return result.text

verification_output = await build_verification_agent(remediation_output)

## Agent 5: Communications Agent

The final agent handles all human communication.

### 🎯 YOUR TASK
Create the Comms Agent. It should:
- Post a resolution summary to Slack
- Create a post-incident ticket
- Update the status page

In [ ]:
async def build_comms_agent(triage_ctx: str, diagnostics_ctx: str, remediation_ctx: str, verification_ctx: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Communications Agent
        comms_instructions = """
You are the Communications Agent. After an incident is resolved, you handle all notifications.

Your responsibilities:
1. Post a concise incident summary to Slack (#incidents channel)
2. Create a post-incident ticket with full details for the post-mortem
3. Update the status page to 'operational' if resolved

Your Slack message should be brief and include:
- What happened (1 sentence)
- Root cause (1 sentence)
- Fix applied (1 sentence)
- Current status (resolved/monitoring)

The incident ticket should have the full timeline and details.
"""

        comms_agent = client.create_agent(
            name="CommunicationsAgent",
            instructions=comms_instructions,
            tools=[post_to_slack, create_incident_ticket, update_status_page],
        )

        comms_prompt = f"""The incident has been resolved. Notify the team.

--- FULL INCIDENT TIMELINE ---

TRIAGE:
{triage_ctx}

DIAGNOSTICS:
{diagnostics_ctx}

REMEDIATION:
{remediation_ctx}

VERIFICATION:
{verification_ctx}

--- SERVICE ---
{alert['service']}

Post to Slack, create an incident ticket, and update the status page."""

        result = await comms_agent.run(comms_prompt)
        print("\n📢 COMMUNICATIONS RESULT:\n")
        print(result.text)
        return result.text

comms_output = await build_comms_agent(triage_output, diagnostics_output, remediation_output, verification_output)

## 🎉 What We've Built

We now have 5 specialized agents, each with focused tools:

```
Alert ──► Triage ──► Diagnostics ──► Remediation ──► Verification ──► Comms
          (2 tools)   (3 tools)       (5 tools)       (2 tools)       (3 tools)
```

**But there's a problem...**

We're manually passing outputs between agents. What if:
- The verification fails? We need to loop back to remediation.
- We want to run this without human intervention?
- We want to handle different incident types with different routing?

That's where **workflow orchestration** comes in.

---

## ➡️ Next: Step 3 — Workflow Orchestration

In the next notebook, we'll wire these agents into an automated `WorkflowBuilder` that:
- Chains agents together automatically
- Routes based on conditions (fix worked → comms, fix failed → retry/escalate)
- Runs the entire pipeline with one call